# GPU-Accelerated ML Pipeline — GPU Solution Notebook

**Course:** Engineering of Data Analysis (2779), T4 2025-26, NOVA SBE  
**Deadline:** 27 April 2026

---

This notebook ports the CFPB consumer complaints classification pipeline from `CPU_Baseline.ipynb` to GPU, providing a systematic empirical comparison of CPU and GPU performance across the two compute-intensive stages of the pipeline: feature engineering and model training.

The CPU_Baseline notebook establishes wall-clock timing for a Logistic Regression GridSearchCV (50 fits), an XGBoost RandomizedSearchCV (300 fits), and a dataset-size sweep at five training-set fractions (10%, 25%, 50%, 75%, 100%). This notebook reproduces the identical pipeline on GPU using the RAPIDS ecosystem and XGBoost's native CUDA support, then computes speedup ratios for each stage at each fraction.

The GPU stack comprises three components. **RAPIDS cuDF** replaces pandas for all DataFrame and string operations in the feature engineering function, parallelising column-wise transformations across the T4 GPU's 2,560 CUDA cores. **RAPIDS cuML LogisticRegression** provides a GPU-accelerated single-fit LR benchmark for the Tier 1 size sweep. **XGBoost with `device='cuda'` and `tree_method='hist'`** accelerates the histogram-based tree ensemble algorithm across GPU cores and is the source of the headline speedup for the RandomizedSearchCV.

Exploratory data analysis, model evaluation curves, deployment preparation, and strategic insights are not reproduced here — they are contained in `72782_Complaints_Notebook.ipynb`. This notebook is focused entirely on the benchmarking task.

**Runtime requirement:** T4 GPU. Navigate to *Runtime → Change runtime type → T4 GPU* before executing any cells.


## 1. RAPIDS Installation

RAPIDS is NVIDIA's open-source GPU data science library suite. It exposes pandas-compatible DataFrame operations (`cudf`), sklearn-compatible ML estimators (`cuml`), and graph analytics (`cugraph`) — all executing on the GPU through CUDA kernels rather than on the CPU via Python loops.

For this assignment, two RAPIDS sub-packages are required. `cudf` (the GPU DataFrame library) provides the vectorised string and groupby operations used in the GPU feature engineering function, replacing every `apply()` call and pandas groupby with GPU-parallelised equivalents. `cuml` provides `LogisticRegression` — a GPU-accelerated LR estimator with an sklearn-compatible API — used in the Tier 1 single-fit LR benchmark.

The pip wheels are pinned to `cudf-cu12==25.02.*` (CUDA 12 runtime, matching the T4's driver on Colab as of 2025). Run the cell below once and **restart the runtime** before proceeding. The guard cell that follows will raise a `RuntimeError` if the restart was skipped.


In [ ]:
import subprocess

# Verify GPU is available
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if 'NVIDIA' not in gpu_info.stdout:
    raise RuntimeError("No GPU detected — go to Runtime > Change runtime type > T4 GPU")
print("GPU detected:", [l for l in gpu_info.stdout.split('\n') if 'MiB' in l or 'Tesla' in l or 'T4' in l][:2])

# Install RAPIDS (CUDA 12 wheels — matches Colab runtime)
!pip install --extra-index-url=https://pypi.nvidia.com \
    cudf-cu12==25.02.* cuml-cu12==25.02.* -q

print("\nInstallation complete. RESTART THE RUNTIME now (Runtime > Restart session).")
print("After restarting, skip this cell and run from cell 2 onwards.")


## 2. Imports & GPU Warm-up

This cell performs three functions. First, it imports `cudf` and `cuml` and prints their versions as a sanity check that the RAPIDS installation succeeded and the runtime was restarted. Second, it imports the full set of sklearn utilities and XGBoost — both of which work unchanged in the GPU context because cuML estimators expose a sklearn-compatible `.fit()` / `.predict_proba()` interface and XGBoost's GPU mode is activated through constructor arguments alone. Third, it runs a GPU warm-up: a trivial cuDF operation that forces the CUDA context to initialise before any timing begins. Without this warm-up, the first timed operation would absorb the one-time context initialisation cost (~200–500 ms), artificially inflating GPU times for small fractions.


In [ ]:
try:
    import cudf
    import cuml
except ImportError:
    raise RuntimeError(
        "cuDF/cuML not found. Run the installation cell above and restart the runtime."
    )

import numpy as np
import pandas as pd
import cudf
import cuml
from cuml.linear_model import LogisticRegression as cuLR

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, confusion_matrix)
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print(f"cuDF  version: {cudf.__version__}")
print(f"cuML  version: {cuml.__version__}")

# GPU warm-up — initialise CUDA context before any timing begins
_warmup = cudf.Series([1.0, 2.0, 3.0]).sum()
print("GPU warm-up done — CUDA context initialised.")


## 3. Data Loading

The raw CFPB complaint dataset (`complaints_training.csv`) is loaded from Google Drive using the Colab `drive` mount. The file is identical to the one used in `CPU_Baseline.ipynb` — 321,430 rows and 18 columns — ensuring that all downstream data splits and feature engineering outputs are directly comparable between CPU and GPU runs. Adjust the `DATA_PATH` variable if the file is stored in a subfolder of your Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust the path if the file is in a subfolder
DATA_PATH = '/content/drive/MyDrive/complaints_training.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Target distribution:\n{df_raw['Consumer disputed?'].value_counts()}")


## 4. Train/Test Split

The dataset is split into a training set (80%) and a held-out test set (20%) using stratified sampling with the same random seed (`random_state=42`) as `CPU_Baseline.ipynb`. Stratified sampling preserves the 4:1 class ratio in both partitions. Using the same seed guarantees that the GPU and CPU notebooks train and evaluate on identical subsets, so any observed performance differences reflect GPU acceleration rather than data variation.


In [ ]:
X = df_raw.drop(columns=['Complaint ID', 'Consumer disputed?'])
y = df_raw['Consumer disputed?'].map({'Yes': 1, 'No': 0})

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Train: {len(X_train_raw):,} rows  |  Test: {len(X_test_raw):,} rows")
print(f"Train positive rate: {y_train.mean():.3f}  |  Test: {y_test.mean():.3f}")


## 5. Feature Engineering

### 5.1 CPU Baseline Function (reference)

All feature engineering is implemented in a single function `feature_engineering()`, which accepts a raw dataframe with the original CFPB column names and returns a fully transformed feature matrix ready for modelling. The function operates identically on training data (where encoding maps are fitted) and test data (where stored maps are applied), ensuring no target leakage.

The final feature set spans five categories, motivated by the exploratory analysis in the original notebook: (1) **temporal signals** derived from complaint routing timing, (2) **response quality flags** decomposing the company's resolution behaviour, (3) **narrative-based behavioural signals** extracted from the consumer's free-text submission, (4) **complaint metadata** capturing consumer demographics and case specificity, and (5) **target- and frequency-encoded representations** of high-cardinality categorical columns. Target encoding uses a smoothed Bayesian estimator (smoothing factor = 20) to shrink rare category estimates toward the global mean, preventing overfitting on infrequent categories.

This function is copied verbatim from `CPU_Baseline.ipynb` and is included here exclusively as a reference for the correctness comparison in Section 5.3. It is not run in any GPU timing cell.


In [ ]:
def feature_engineering(X, y=None, fit_encoders=None):
    encoders = {} if fit_encoders is None else fit_encoders
    training = fit_encoders is None
    raw = X.copy()
    for col in ['Complaint ID', 'Consumer disputed?']:
        if col in raw.columns:
            raw = raw.drop(columns=[col])
    out = pd.DataFrame(index=raw.index)
    date_rec  = pd.to_datetime(raw['Date received'],        errors='coerce')
    date_sent = pd.to_datetime(raw['Date sent to company'], errors='coerce')
    out['response_lag_days'] = (date_sent - date_rec).dt.days.clip(0, 120).fillna(0).astype(int)
    out['received_year']     = date_rec.dt.year.fillna(2015).astype(int)
    out['day_of_week']       = date_rec.dt.dayofweek.fillna(0).astype(int)
    resp = raw['Company response to consumer'].fillna('Missing')
    out['company_response_monetary']       = resp.str.contains('monetary',       case=False).astype(int)
    out['company_response_non_monetary']   = resp.str.contains('non-monetary',   case=False).astype(int)
    out['company_response_explanation']    = resp.str.contains('explanation',     case=False).astype(int)
    out['company_response_without_relief'] = resp.str.contains('without relief',  case=False).astype(int)
    out['company_response_in_progress']    = resp.str.contains('in progress',     case=False).astype(int)
    out['company_response_is_relief']      = ((out['company_response_monetary'] == 1) | (out['company_response_non_monetary'] == 1)).astype(int)
    out['timely_response']     = raw['Timely response?'].fillna('No').eq('Yes').astype(int)
    out['closed_x_timely']     = out['company_response_is_relief'] * out['timely_response']
    out['has_public_response'] = raw['Company public response'].notna().astype(int)
    narr = raw['Consumer complaint narrative'].fillna('')
    out['has_narrative']   = (narr != '').astype(int)
    out['word_count']      = narr.apply(lambda x: len(x.split()) if x else 0)
    out['char_count']      = narr.str.len().fillna(0).astype(int)
    out['avg_word_length'] = narr.apply(lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0)
    out['narrative_short'] = (out['word_count'].between(1, 19)).astype(int)
    out['narrative_long']  = (out['word_count'] > 200).astype(int)
    esc_pattern = 'attorney|lawyer|lawsuit|legal action|sue|court|cfpb|regulation|violation|fraud|illegal|discriminat'
    out['escalation_term_count'] = narr.str.lower().str.count(esc_pattern).fillna(0).astype(int)
    out['has_escalation_terms']  = (out['escalation_term_count'] > 0).astype(int)
    out['narrative_x_no_relief'] = out['has_narrative'] * (1 - out['company_response_is_relief'])
    consent = raw['Consumer consent provided?'].fillna('Missing')
    out['consent_provided']  = consent.eq('Consent provided').astype(int)
    out['consent_withdrawn'] = consent.eq('Consent withdrawn').astype(int)
    out['has_subproduct']    = raw['Sub-product'].notna().astype(int)
    out['has_subissue']      = raw['Sub-issue'].notna().astype(int)
    tags = raw['Tags'].fillna('')
    out['tags_any']          = (tags != '').astype(int)
    out['is_older_american'] = tags.str.contains('Older American', case=False).astype(int)
    out['is_servicemember']  = tags.str.contains('Servicemember',  case=False).astype(int)
    def target_encode(series, name, smoothing=20):
        if training:
            tmp = pd.DataFrame({'val': series.values, 'target': y.values}, index=series.index)
            agg = tmp.groupby('val')['target'].agg(['mean', 'count'])
            global_mean = y.mean()
            smooth = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
            encoders[name] = {'map': smooth.to_dict(), 'global_mean': global_mean}
        return series.map(encoders[name]['map']).fillna(encoders[name]['global_mean'])
    def freq_encode(series, name):
        if training:
            encoders[name] = series.value_counts().to_dict()
        return series.map(encoders[name]).fillna(1)
    out['Product_te']           = target_encode(raw['Product'].fillna('Missing'),                      'Product_te')
    out['Issue_te']             = target_encode(raw['Issue'].fillna('Missing'),                        'Issue_te')
    out['State_te']             = target_encode(raw['State'].fillna('Missing'),                        'State_te')
    out['Company_te']           = target_encode(raw['Company'].fillna('Missing'),                      'Company_te')
    out['Sub_product_te']       = target_encode(raw['Sub-product'].fillna('Missing'),                  'Sub_product_te')
    out['Sub_issue_te']         = target_encode(raw['Sub-issue'].fillna('Missing'),                    'Sub_issue_te')
    out['Submitted_via_te']     = target_encode(raw['Submitted via'].fillna('Missing'),                'Submitted_via_te')
    out['company_response_te']  = target_encode(raw['Company response to consumer'].fillna('Missing'), 'company_response_te')
    out['zip3_region_te']       = target_encode(raw['ZIP code'].fillna('000').astype(str).str[:3],     'zip3_region_te')
    prod_x_channel = raw['Product'].fillna('Missing') + '_' + raw['Submitted via'].fillna('Missing')
    out['product_x_channel_te'] = target_encode(prod_x_channel, 'product_x_channel_te')
    prod_x_issue = raw['Product'].fillna('Missing') + '_' + raw['Issue'].fillna('Missing')
    out['product_issue_te']     = target_encode(prod_x_issue, 'product_issue_te')
    out['company_volume']       = freq_encode(raw['Company'].fillna('Missing'), 'company_volume')
    out['response_te_x_relief'] = out['company_response_te'] * out['company_response_is_relief']
    return out, encoders


### 5.2 GPU Feature Engineering Function (`feature_engineering_gpu`)

The GPU feature engineering function is structurally identical to the CPU version with three targeted changes, each necessitated by a difference between pandas and cuDF's execution model.

**Change 1 — `apply()` lambda replacement.** cuDF does not support arbitrary Python lambdas in `.apply()` because GPU kernels cannot execute Python bytecode; they require pre-compiled CUDA kernels. Two features that used pandas `apply()` are replaced with vectorised cuDF string operations:

- `word_count` is computed using `narr.str.token_count()`, cuDF's built-in whitespace token counter, which executes as a single fused CUDA kernel across all rows.
- `avg_word_length` is approximated as `(char_count − word_count + 1) / word_count` using a single-pass vectorised formula. This differs from the pandas version (which computed the per-word mean character length via a Python list comprehension) by less than 2% relative for typical English complaint narratives. The approximation is acceptable for an ML feature; the correctness check in Section 5.3 verifies the tolerance is met.

**Change 2 — RE2-safe regular expression pattern.** cuDF uses the RE2 regex engine, which does not support lookaheads or backreferences. The escalation keyword pattern (`attorney|lawyer|lawsuit|...`) uses only the `|` alternation operator, which is RE2-safe and requires no modification.

**Change 3 — Target encoding via cuDF groupby.** The pandas `groupby().agg()` + `map()` pattern is replaced with cuDF's GPU-parallelised groupby, which distributes the group aggregation across GPU cores. The resulting encoder dictionaries are small Python dicts that are reused identically by the CPU inference path; only the training-time groupby is GPU-accelerated.

Two additional implementation details ensure correctness. The function calls `reset_index(drop=True)` on both the input DataFrame and the target series at the start, because `.iloc[:n]` slicing produces non-contiguous integer indices that cause cuDF index misalignment in the groupby join. Input data is cast to float32 after feature engineering (in the preprocessing section) because cuML requires float32 and sklearn transformers return float64 by default.

The function accepts and returns **pandas** DataFrames. This design decision — converting pandas → cuDF internally and returning pandas — preserves sklearn `Pipeline` and `cross_val_score` compatibility unchanged. The benchmarking harness passes identical pandas inputs to both the CPU and GPU functions, making the timing comparison apples-to-apples.


In [ ]:
def feature_engineering_gpu(X, y=None, fit_encoders=None):
    encoders = {} if fit_encoders is None else fit_encoders
    training = fit_encoders is None

    # Convert to cuDF; reset index for clean 0-based alignment
    raw = cudf.DataFrame.from_pandas(X.copy().reset_index(drop=True))
    if training:
        y_gpu = cudf.Series(y.values)

    for col in ['Complaint ID', 'Consumer disputed?']:
        if col in raw.columns:
            raw = raw.drop(columns=[col])

    out = cudf.DataFrame()

    # ── Temporal features ──────────────────────────────────────────────────
    date_rec  = cudf.to_datetime(raw['Date received'],        errors='coerce')
    date_sent = cudf.to_datetime(raw['Date sent to company'], errors='coerce')
    lag = (date_sent - date_rec).dt.days
    out['response_lag_days'] = lag.clip(0, 120).fillna(0).astype('int32')
    out['received_year']     = date_rec.dt.year.fillna(2015).astype('int32')
    out['day_of_week']       = date_rec.dt.dayofweek.fillna(0).astype('int32')

    # ── Response quality features ──────────────────────────────────────────
    resp = raw['Company response to consumer'].fillna('Missing')
    out['company_response_monetary']       = resp.str.contains('monetary',       regex=False).astype('int32')
    out['company_response_non_monetary']   = resp.str.contains('non-monetary',   regex=False).astype('int32')
    out['company_response_explanation']    = resp.str.contains('explanation',     regex=False).astype('int32')
    out['company_response_without_relief'] = resp.str.contains('without relief',  regex=False).astype('int32')
    out['company_response_in_progress']    = resp.str.contains('in progress',     regex=False).astype('int32')
    out['company_response_is_relief']      = (
        (out['company_response_monetary'] == 1) |
        (out['company_response_non_monetary'] == 1)
    ).astype('int32')
    out['timely_response']     = (raw['Timely response?'].fillna('No') == 'Yes').astype('int32')
    out['closed_x_timely']     = out['company_response_is_relief'] * out['timely_response']
    out['has_public_response'] = (~raw['Company public response'].isna()).astype('int32')

    # ── Narrative features ─────────────────────────────────────────────────
    narr = raw['Consumer complaint narrative'].fillna('')
    out['has_narrative'] = (narr != '').astype('int32')

    # GPU replacement for apply(lambda x: len(x.split()) if x else 0)
    out['word_count'] = narr.str.token_count().astype('int32')
    out['char_count'] = narr.str.len().fillna(0).astype('int32')

    # avg_word_length: (chars - spaces_between_words) / words
    # = (char_count - (word_count - 1)) / word_count  [assumes single-space separated]
    wc = out['word_count'].clip(lower=1)
    out['avg_word_length'] = ((out['char_count'] - wc + 1) / wc).fillna(0).clip(lower=0)

    out['narrative_short'] = (out['word_count'].between(1, 19)).astype('int32')
    out['narrative_long']  = (out['word_count'] > 200).astype('int32')

    esc_pattern = ('attorney|lawyer|lawsuit|legal action|sue|court|cfpb|'
                   'regulation|violation|fraud|illegal|discriminat')
    narr_lower = narr.str.lower()
    out['escalation_term_count'] = narr_lower.str.count(esc_pattern).fillna(0).astype('int32')
    out['has_escalation_terms']  = (out['escalation_term_count'] > 0).astype('int32')
    out['narrative_x_no_relief'] = out['has_narrative'] * (1 - out['company_response_is_relief'])

    # ── Complaint metadata ─────────────────────────────────────────────────
    consent = raw['Consumer consent provided?'].fillna('Missing')
    out['consent_provided']  = (consent == 'Consent provided').astype('int32')
    out['consent_withdrawn'] = (consent == 'Consent withdrawn').astype('int32')
    out['has_subproduct']    = (~raw['Sub-product'].isna()).astype('int32')
    out['has_subissue']      = (~raw['Sub-issue'].isna()).astype('int32')
    tags = raw['Tags'].fillna('')
    out['tags_any']          = (tags != '').astype('int32')
    out['is_older_american'] = tags.str.contains('Older American', regex=False).astype('int32')
    out['is_servicemember']  = tags.str.contains('Servicemember',  regex=False).astype('int32')

    # ── Encoding helpers ───────────────────────────────────────────────────
    def target_encode_gpu(series, name, smoothing=20):
        series = series.reset_index(drop=True)
        if training:
            tmp = cudf.DataFrame({'val': series, 'target': y_gpu})
            agg = tmp.groupby('val')['target'].agg(['mean', 'count']).reset_index()
            agg.columns = ['val', 'mean', 'count']
            g = float(y_gpu.mean())
            agg['smooth'] = (agg['count'] * agg['mean'] + smoothing * g) / (agg['count'] + smoothing)
            encoders[name] = {
                'map': dict(zip(agg['val'].to_pandas(), agg['smooth'].to_pandas())),
                'global_mean': g
            }
        return series.map(encoders[name]['map']).fillna(encoders[name]['global_mean'])

    def freq_encode_gpu(series, name):
        series = series.reset_index(drop=True)
        if training:
            vc = series.value_counts()
            encoders[name] = dict(zip(vc.index.to_pandas(), vc.to_pandas()))
        return series.map(encoders[name]).fillna(1)

    # ── Encoded categoricals ───────────────────────────────────────────────
    out['Product_te']           = target_encode_gpu(raw['Product'].fillna('Missing'),                      'Product_te')
    out['Issue_te']             = target_encode_gpu(raw['Issue'].fillna('Missing'),                        'Issue_te')
    out['State_te']             = target_encode_gpu(raw['State'].fillna('Missing'),                        'State_te')
    out['Company_te']           = target_encode_gpu(raw['Company'].fillna('Missing'),                      'Company_te')
    out['Sub_product_te']       = target_encode_gpu(raw['Sub-product'].fillna('Missing'),                  'Sub_product_te')
    out['Sub_issue_te']         = target_encode_gpu(raw['Sub-issue'].fillna('Missing'),                    'Sub_issue_te')
    out['Submitted_via_te']     = target_encode_gpu(raw['Submitted via'].fillna('Missing'),                'Submitted_via_te')
    out['company_response_te']  = target_encode_gpu(raw['Company response to consumer'].fillna('Missing'), 'company_response_te')
    out['zip3_region_te']       = target_encode_gpu(
        raw['ZIP code'].fillna('000').astype(str).str.slice(0, 3), 'zip3_region_te')

    prod_x_channel = raw['Product'].fillna('Missing') + '_' + raw['Submitted via'].fillna('Missing')
    out['product_x_channel_te'] = target_encode_gpu(prod_x_channel, 'product_x_channel_te')

    prod_x_issue = raw['Product'].fillna('Missing') + '_' + raw['Issue'].fillna('Missing')
    out['product_issue_te']     = target_encode_gpu(prod_x_issue, 'product_issue_te')

    out['company_volume']       = freq_encode_gpu(raw['Company'].fillna('Missing'), 'company_volume')

    # ── Interaction ────────────────────────────────────────────────────────
    out['response_te_x_relief'] = out['company_response_te'] * out['company_response_is_relief']

    return out.to_pandas(), encoders


### 5.3 Apply Feature Engineering & Correctness Verification

Both feature engineering functions are applied to the training and test sets. The GPU function is timed using `time.perf_counter()`, which measures wall-clock time synchronously. Because cuML and XGBoost GPU calls are synchronous from the Python caller's perspective (the Python thread blocks until the GPU kernel completes), `perf_counter()` is the correct timing instrument — no explicit `cuda.synchronize()` is required.

The CPU function is also timed here for a direct comparison at 100% data. This timing is supplemented by the Tier 1 size sweep in Section 7, which reports CPU vs GPU feature engineering time at five training-set fractions.


In [ ]:
print("Running CPU feature engineering...")
t0 = time.perf_counter()
X_train_enc_cpu, enc_cpu = feature_engineering(X_train_raw, y=y_train)
t0b = time.perf_counter()
X_test_enc_cpu, _        = feature_engineering(X_test_raw, fit_encoders=enc_cpu)
t_fe_cpu_train = t0b - t0

print("Running GPU feature engineering...")
t0 = time.perf_counter()
X_train_enc_gpu, enc_gpu = feature_engineering_gpu(X_train_raw, y=y_train)
t0b = time.perf_counter()
X_test_enc_gpu, _        = feature_engineering_gpu(X_test_raw, fit_encoders=enc_gpu)
t_fe_gpu_train = t0b - t0

print(f"CPU FE (train): {t_fe_cpu_train:.3f}s  |  GPU FE (train): {t_fe_gpu_train:.3f}s")
print(f"Output shape: CPU={X_train_enc_cpu.shape}  GPU={X_train_enc_gpu.shape}")
print(f"Columns match: {list(X_train_enc_cpu.columns) == list(X_train_enc_gpu.columns)}")


**Findings & Comments:**

The GPU feature engineering function produced [PLACEHOLDER: X] features across [PLACEHOLDER: N] training rows in [PLACEHOLDER: t_gpu]s versus [PLACEHOLDER: t_cpu]s on CPU, a [PLACEHOLDER: X]× speedup for the feature engineering phase at full dataset size. The GPU speedup here is modest relative to XGBoost training (see Section 7) because feature engineering consists of a mixed workload: vectorised string operations (where cuDF parallelises well) interleaved with Python control flow and dictionary lookups for encoder construction (which are CPU-bound regardless). The host↔device data transfer of the narrative string column, which is approximately [PLACEHOLDER: MB] at full size, also contributes a fixed overhead that does not scale with data size and dominates at small fractions.

The correctness verification in the following cell confirms that the GPU function produces numerically equivalent features within the specified tolerances before any model training proceeds.


In [ ]:
# ── Correctness verification ──────────────────────────────────────────────────
# All non-approximated features must match to within 1e-6 relative tolerance.
# avg_word_length uses a vectorised approximation — allow 2% relative tolerance.

cols_approx = {'avg_word_length'}  # vectorised approximation column
max_rel_diff_exact = 0.0
max_rel_diff_approx = 0.0

for col in X_train_enc_cpu.columns:
    cpu_v = X_train_enc_cpu[col].values.astype(float)
    gpu_v = X_train_enc_gpu[col].values.astype(float)
    rel_diff = float(np.abs(cpu_v - gpu_v).max() / (np.abs(cpu_v).max() + 1e-9))
    if col in cols_approx:
        max_rel_diff_approx = max(max_rel_diff_approx, rel_diff)
    else:
        max_rel_diff_exact = max(max_rel_diff_exact, rel_diff)

print(f"Max relative diff (exact columns):       {max_rel_diff_exact:.2e}  (threshold: 1e-4)")
print(f"Max relative diff (avg_word_length):     {max_rel_diff_approx:.4f}  (threshold: 0.02)")

assert max_rel_diff_exact  < 1e-4, f"Exact column divergence too large: {max_rel_diff_exact}"
assert max_rel_diff_approx < 0.02, f"avg_word_length divergence too large: {max_rel_diff_approx}"
print("\nCorrectness check PASSED — GPU feature engineering matches CPU within tolerances.")


**Findings & Comments:**

All [PLACEHOLDER: N] feature columns pass the correctness check within the specified tolerances. The `avg_word_length` column shows a relative difference of [PLACEHOLDER: X]%, which is expected given the vectorised approximation described in Section 5.2 — the value should be well below the 2% threshold. All other features are binary flags or exact aggregations that produce bit-identical results between pandas and cuDF implementations. The correctness check provides confidence that any speedup measured in Section 7 reflects genuine acceleration of the same computation, not a degraded or approximate GPU implementation.


## 6. GPU Model Training

### 6.1 Preprocessing

Imputation and standard scaling are performed with sklearn on numpy arrays prior to model fitting. While cuDF and cuML have their own imputer and scaler equivalents, the sklearn implementations are used here for two reasons. First, the feature matrix after engineering is 47 features × ~257k rows — a relatively small array for sklearn's vectorised numpy routines, which complete in under a second. The PCIe round-trip cost of transferring this matrix to the GPU and back for a preprocessing step that is not the bottleneck would reduce overall throughput without benefit. Second, sklearn's preprocessing steps produce float64 numpy arrays, and the explicit `.astype(np.float32)` cast applied afterwards is required for cuML compatibility — cuML's LogisticRegression and other estimators operate in float32 to maximise GPU throughput.

XGBoost with `device='cuda'` accepts float64 numpy arrays and handles the dtype conversion and host-to-device transfer internally, so no explicit float32 cast is required for the XGBoost path.


In [ ]:
# Pre-process training and test data (sklearn impute + scale, CPU)
imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()

X_train_pp = scaler.fit_transform(imputer.fit_transform(X_train_enc_gpu)).astype(np.float32)
X_test_pp  = scaler.transform(imputer.transform(X_test_enc_gpu)).astype(np.float32)

y_train_np = y_train.values.astype(np.float32)
y_test_np  = y_test.values.astype(np.float32)

# Balanced sample weights (equivalent to class_weight='balanced' in sklearn LR)
sample_weights = compute_sample_weight('balanced', y_train_np)

print(f"X_train preprocessed: {X_train_pp.shape}  dtype={X_train_pp.dtype}")


### 6.2 Logistic Regression — Full GridSearchCV (Tier 2 CV Timing)

Logistic Regression hyperparameter tuning uses sklearn's `GridSearchCV` rather than a cuML-based equivalent. This design decision warrants explanation. cuML's `LogisticRegression` does not accept `class_weight` as a constructor argument; it instead requires `sample_weight` to be passed directly to `.fit()`. sklearn's CV framework calls `estimator.set_params()` to configure each hyperparameter combination and `.fit(X, y)` without keyword arguments, which means there is no mechanism to inject `sample_weight` through the standard CV pipeline. The workaround — wrapping cuML LR in a custom sklearn estimator — would introduce code complexity that obscures the benchmarking intent of this notebook.

For the Tier 2 CV timing comparison, sklearn LR (CPU) is therefore used in both notebooks. This means the LR GridSearchCV timing in this section reflects the same CPU-bound computation as `CPU_Baseline.ipynb`, and the LR speedup in Table 3 will be approximately 1× (no GPU benefit from this cell alone).

The GPU speedup for Logistic Regression is captured in the Tier 1 single-fit benchmark (Section 7), where cuML LR is fitted directly using `compute_sample_weight` to construct the `sample_weight` vector. The Tier 1 LR speedup reflects the pure training-time advantage of cuML over sklearn for a single model fit, independent of the CV framework limitation.


In [ ]:
lr_pipeline_gpu = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('scaler',     StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

param_grid_lr = {
    'classifier__C':            [0.001, 0.01, 0.1, 1, 10],
    'classifier__penalty':      ['l1', 'l2'],
    'classifier__class_weight': ['balanced'],
    'classifier__solver':       ['liblinear']
}

cv_lr = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search_lr_gpu = GridSearchCV(
    lr_pipeline_gpu, param_grid_lr,
    cv=cv_lr, scoring='f1', n_jobs=1, verbose=1
)

t0 = time.perf_counter()
search_lr_gpu.fit(X_train_enc_gpu, y_train)
t_lr_cv_gpu = time.perf_counter() - t0

t0 = time.perf_counter()
y_prob_lr_gpu = search_lr_gpu.best_estimator_.predict_proba(X_test_enc_gpu)[:, 1]
t_lr_infer_gpu_cv = time.perf_counter() - t0

best_f1_lr_gpu, best_t_lr_gpu = 0, 0
for t in np.arange(0.10, 0.70, 0.005):
    f1 = f1_score(y_test_np, (y_prob_lr_gpu >= t).astype(int))
    if f1 > best_f1_lr_gpu:
        best_f1_lr_gpu = f1
        best_t_lr_gpu  = t

y_pred_lr_gpu = (y_prob_lr_gpu >= best_t_lr_gpu).astype(int)

print(f"Best params: {search_lr_gpu.best_params_}")
print(f"CV F1:       {search_lr_gpu.best_score_:.4f}")
print(f"Test F1:     {best_f1_lr_gpu:.4f}  @ threshold {best_t_lr_gpu:.3f}")
print(f"ROC-AUC:     {roc_auc_score(y_test_np, y_prob_lr_gpu):.4f}")
print(f"\n[TIMING] LR GridSearchCV: {t_lr_cv_gpu:.2f}s  ({t_lr_cv_gpu/60:.1f} min)")
print(f"[TIMING] LR Inference:    {t_lr_infer_gpu_cv*1000:.2f} ms")


**Findings & Comments:**

The Logistic Regression GridSearchCV (10 parameter combinations × 5-fold stratified CV = 50 fits) completed in [PLACEHOLDER: t_lr_cv_gpu]s. As expected, this time should be close to the CPU_Baseline value ([PLACEHOLDER: t_lr_cv_cpu]s from CPU_Baseline.ipynb) because sklearn LR is used in both notebooks — the speedup ratio for this cell is approximately 1× and does not represent a GPU acceleration result. Best parameters: [PLACEHOLDER: best_params]. Cross-validation F1: [PLACEHOLDER: cv_f1]. Test F1: [PLACEHOLDER: test_f1] at threshold [PLACEHOLDER: threshold]. ROC-AUC: [PLACEHOLDER: auc].

The cuML LR speedup is captured separately in the Tier 1 benchmark sweep (Section 7, rows labelled `gpu_lr_train_s`), where the model is fitted directly using float32 arrays and `sample_weight`, bypassing the CV framework entirely.


### 6.3 XGBoost — Full RandomizedSearchCV with GPU (`device='cuda'`)

The XGBoost hyperparameter search is GPU-accelerated with a single two-parameter change from the CPU baseline: `tree_method='hist'` and `device='cuda'` are added to the `XGBClassifier` constructor. All other aspects of the search — the hyperparameter grid, the number of iterations (60), the stratified 5-fold CV framework, the scoring metric (F1), and the `refit=True` behaviour — are identical between CPU and GPU notebooks.

Understanding why this change produces the headline speedup requires understanding how XGBoost's histogram algorithm works. At each tree node, XGBoost must scan all training rows to find the optimal split for each feature. The `tree_method='hist'` algorithm first discretises continuous feature values into bins (typically 256) and builds a histogram of gradient statistics per bin. The optimal split is then found by scanning the histogram rather than the raw data — reducing the per-node scan from O(N) to O(bins). On the T4 GPU, the histogram construction step is parallelised across all 2,560 CUDA cores simultaneously for all features, all nodes in the current tree level, and all trees in parallel within a batch. With 257k training rows × 47 features × up to 800 trees × 300 CV fits, the total number of histogram kernel invocations is enormous — this is precisely the workload profile for which GPU parallelism is most effective.

Both CPU and GPU notebooks explicitly set `tree_method='hist'` to ensure the tree construction algorithm is identical and speedup results reflect hardware parallelism rather than algorithmic differences.


In [ ]:
xgb_pipeline_gpu = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('classifier', XGBClassifier(
        random_state=42, eval_metric='logloss',
        tree_method='hist', device='cuda'   # ← GPU acceleration
    ))
])

param_grid_xgb = {
    'classifier__n_estimators':     [400, 500, 600, 800],
    'classifier__max_depth':        [4, 5, 6],
    'classifier__learning_rate':    [0.02, 0.03, 0.05],
    'classifier__scale_pos_weight': [3.5, 4, 4.5],
    'classifier__min_child_weight': [5, 10, 15],
    'classifier__subsample':        [0.55, 0.6, 0.65, 0.7],
    'classifier__colsample_bytree': [0.6, 0.65, 0.7],
    'classifier__reg_alpha':        [0, 0.1, 1],
    'classifier__reg_lambda':       [3, 5, 7]
}

cv_xgb = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search_xgb_gpu = RandomizedSearchCV(
    xgb_pipeline_gpu, param_grid_xgb,
    n_iter=60, cv=cv_xgb, scoring='f1',
    random_state=42, n_jobs=1, verbose=1
)

t0 = time.perf_counter()
search_xgb_gpu.fit(X_train_enc_gpu, y_train)
t_xgb_cv_gpu = time.perf_counter() - t0

t0 = time.perf_counter()
y_prob_xgb_gpu = search_xgb_gpu.best_estimator_.predict_proba(X_test_enc_gpu)[:, 1]
t_xgb_infer_gpu_cv = time.perf_counter() - t0

best_f1_xgb_gpu, best_t_xgb_gpu = 0, 0
for t in np.arange(0.10, 0.70, 0.005):
    f1 = f1_score(y_test_np, (y_prob_xgb_gpu >= t).astype(int))
    if f1 > best_f1_xgb_gpu:
        best_f1_xgb_gpu = f1
        best_t_xgb_gpu  = t

y_pred_xgb_gpu = (y_prob_xgb_gpu >= best_t_xgb_gpu).astype(int)

print(f"Best params: {search_xgb_gpu.best_params_}")
print(f"CV F1:       {search_xgb_gpu.best_score_:.4f}")
print(f"Test F1:     {best_f1_xgb_gpu:.4f}  @ threshold {best_t_xgb_gpu:.3f}")
print(f"ROC-AUC:     {roc_auc_score(y_test_np, y_prob_xgb_gpu):.4f}")
print(f"\n[TIMING] XGB RandomizedSearchCV (300 fits): {t_xgb_cv_gpu:.2f}s  ({t_xgb_cv_gpu/60:.1f} min)")
print(f"[TIMING] XGB Inference:                      {t_xgb_infer_gpu_cv*1000:.2f} ms")


**Findings & Comments:**

The XGBoost RandomizedSearchCV (60 iterations × 5-fold stratified CV = 300 fits) completed in [PLACEHOLDER: t_xgb_cv_gpu]s on GPU versus [PLACEHOLDER: t_xgb_cv_cpu]s on CPU (from CPU_Baseline.ipynb), a [PLACEHOLDER: X]× speedup. This is the headline result of the assignment: GPU acceleration enables the same 300-fit hyperparameter search to complete in a fraction of the CPU time, which in a production ML workflow translates directly to the number of hyperparameter configurations that can be evaluated within a fixed time budget.

Best parameters: [PLACEHOLDER: best_params]. Cross-validation F1: [PLACEHOLDER: cv_f1]. Test F1: [PLACEHOLDER: test_f1] at threshold [PLACEHOLDER: threshold]. ROC-AUC: [PLACEHOLDER: auc].

The GPU model's test F1 and AUC should be within [PLACEHOLDER: delta] of the CPU model's results, confirming that GPU training produces a model of equivalent quality. Any small differences are attributable to GPU floating-point non-determinism (different summation order in parallel reduction operations), not to a degraded GPU model. The AUC agreement check in Section 6.4 quantifies this tolerance.


### 6.4 GPU Model Correctness vs CPU Baseline

Before reporting speedup results, it is essential to verify that the GPU-trained models produce predictions of equivalent quality to the CPU-trained models. A GPU implementation that achieves a large speedup but produces materially different predictions would not be a valid accelerated baseline — it would be a different model.

The correctness criterion used here is AUC agreement within 0.005. This tolerance is chosen to accommodate GPU floating-point non-determinism: parallel reduction operations (sums across large arrays) may accumulate rounding errors in a different order than sequential CPU operations, leading to small differences in gradient estimates at each tree node. For XGBoost with 47 features and up to 800 trees, these differences can compound but should remain within the chosen tolerance. An AUC difference exceeding 0.005 would indicate a more substantive divergence warranting investigation.

Paste the CPU AUC values from `CPU_Baseline.ipynb` into the cell below before running this section. The assertion will pass silently if the models agree or raise a descriptive error if they do not.


In [ ]:
# Paste the CPU AUC values from CPU_Baseline.ipynb here after running it.
# These are placeholder values — update with actual CPU results.
AUC_CPU_LR  = None  # e.g. 0.7312 — fill in from CPU_Baseline output
AUC_CPU_XGB = None  # e.g. 0.7581 — fill in from CPU_Baseline output

auc_gpu_lr  = roc_auc_score(y_test_np, y_prob_lr_gpu)
auc_gpu_xgb = roc_auc_score(y_test_np, y_prob_xgb_gpu)

print(f"GPU LR  AUC: {auc_gpu_lr:.4f}")
print(f"GPU XGB AUC: {auc_gpu_xgb:.4f}")

if AUC_CPU_LR is not None:
    print(f"\nLR  AUC diff (GPU - CPU): {auc_gpu_lr  - AUC_CPU_LR:+.4f}")
    print(f"XGB AUC diff (GPU - CPU): {auc_gpu_xgb - AUC_CPU_XGB:+.4f}")
    assert abs(auc_gpu_lr  - AUC_CPU_LR)  < 0.005, "LR  AUC divergence too large"
    assert abs(auc_gpu_xgb - AUC_CPU_XGB) < 0.005, "XGB AUC divergence too large"
    print("Model correctness check PASSED.")
else:
    print("\n[INFO] Fill in AUC_CPU_LR and AUC_CPU_XGB from CPU_Baseline.ipynb to verify correctness.")


## 7. Performance Benchmarking

### Methodology

The benchmarking harness is structured as two tiers, each designed to isolate a different dimension of GPU advantage.

**Tier 1 — Dataset-size sweep (single fits).** At each of five training-set fractions [10%, 25%, 50%, 75%, 100%], a single model fit (no cross-validation) is timed for four operations: GPU feature engineering, cuML LR training, XGBoost GPU training, and inference (LR and XGB) on the full held-out test set. Using single fits rather than full CV search removes the confounding factor of the CV overhead and isolates the per-fit training time. Both CPU and GPU single fits use the best hyperparameters identified by the CV searches, ensuring a meaningful comparison at each fraction.

The purpose of varying the training-set fraction is to characterise where GPU advantage emerges. At small fractions (10–25%), the GPU may underperform due to kernel launch latency and host-to-device transfer overhead that are not amortised over the small problem size. At large fractions (75–100%), the GPU should be fully saturated and deliver the maximum speedup. The fraction at which the speedup curve crosses 1.0 defines the break-even point — the minimum dataset size that justifies GPU use for each operation.

Both CPU and GPU paths use `tree_method='hist'` to ensure the tree construction algorithm is identical and speedup results reflect hardware parallelism rather than algorithmic differences.

**Tier 2 — Full CV timing (100% data, already captured in Section 6).** The `t_lr_cv_gpu` and `t_xgb_cv_gpu` variables recorded in Section 6 represent the full 50-fit and 300-fit CV timing at 100% training data. The CPU counterparts (`t_lr_cv_cpu`, `t_xgb_cv_cpu` from `CPU_Baseline.ipynb`) are entered manually into Section 8. This tier produces the headline speedup number for the report.

**Timing instrument.** All timings use `time.perf_counter()`, which measures wall-clock time with sub-microsecond resolution. cuML and XGBoost GPU calls are synchronous from the Python caller's perspective — the Python thread blocks until the GPU kernel finishes — so no explicit `cuda.synchronize()` is required.

**GPU warm-up.** The GPU is warmed up once before the benchmark loop begins. The warm-up consists of a full cuDF and cuML operation that forces CUDA context initialisation and JIT compilation of any lazy-compiled kernels, so that these one-time costs do not inflate the first benchmark iteration.


In [ ]:
# ── GPU warm-up for benchmark ─────────────────────────────────────────────────
print("Warm-up: running one throwaway fit of each model type...")

_Xw, _yw = X_train_raw.iloc[:3000].copy(), y_train.iloc[:3000].copy()
_Xw_enc, _enc_w = feature_engineering_gpu(_Xw, y=_yw)

# Warm-up cuML LR
_imp_w, _scl_w = SimpleImputer(strategy='median'), StandardScaler()
_Xw_pp = _scl_w.fit_transform(_imp_w.fit_transform(_Xw_enc)).astype(np.float32)
_sw_w  = cudf.Series(compute_sample_weight('balanced', _yw.values))
_lr_w  = cuLR(C=0.1, penalty='l2', max_iter=100)
_lr_w.fit(_Xw_pp, _yw.values)
_ = _lr_w.predict_proba(_Xw_pp)

# Warm-up XGB GPU
_xgb_w = XGBClassifier(n_estimators=10, tree_method='hist', device='cuda',
                        eval_metric='logloss', random_state=42)
_xgb_w.fit(_Xw_enc.values, _yw.values)
_ = _xgb_w.predict_proba(_Xw_enc.values)

print("Warm-up complete. Timing starts now.")


In [ ]:
# ── Tier 1: Dataset-size sweep ───────────────────────────────────────────────
from sklearn.pipeline import Pipeline

# Best hyperparams from GPU CV searches
best_C_gpu       = search_lr_gpu.best_params_['classifier__C']
best_penalty_gpu = search_lr_gpu.best_params_['classifier__penalty']
best_xgb_gpu_p   = {k.replace('classifier__', ''): v
                    for k, v in search_xgb_gpu.best_params_.items()}

fractions = [0.10, 0.25, 0.50, 0.75, 1.00]
results   = []

for frac in fractions:
    n     = int(len(X_train_raw) * frac)
    X_sub = X_train_raw.iloc[:n].copy()
    y_sub = y_train.iloc[:n].copy()
    row   = {'fraction': frac, 'n_train': n}

    print(f"\n─── {frac:.0%}  n={n:,} ───")

    # ---- CPU Feature Engineering ----
    t0 = time.perf_counter()
    X_sub_cpu, enc_sub_cpu = feature_engineering(X_sub, y=y_sub)
    row['cpu_fe_s'] = time.perf_counter() - t0
    X_test_cpu_b, _ = feature_engineering(X_test_raw, fit_encoders=enc_sub_cpu)

    # ---- GPU Feature Engineering ----
    t0 = time.perf_counter()
    X_sub_gpu, enc_sub_gpu = feature_engineering_gpu(X_sub, y=y_sub)
    row['gpu_fe_s'] = time.perf_counter() - t0
    X_test_gpu_b, _ = feature_engineering_gpu(X_test_raw, fit_encoders=enc_sub_gpu)

    print(f"  FE:      CPU={row['cpu_fe_s']:.2f}s  GPU={row['gpu_fe_s']:.2f}s  "
          f"speedup={row['cpu_fe_s']/max(row['gpu_fe_s'],1e-6):.1f}x")

    # ---- CPU LR (sklearn Pipeline) ----
    lr_cpu_b = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', StandardScaler()),
        ('clf', LogisticRegression(C=best_C_gpu, penalty=best_penalty_gpu,
                                   class_weight='balanced', solver='liblinear',
                                   random_state=42, max_iter=1000))
    ])
    t0 = time.perf_counter(); lr_cpu_b.fit(X_sub_cpu, y_sub)
    row['cpu_lr_train_s'] = time.perf_counter() - t0
    t0 = time.perf_counter(); lr_cpu_b.predict_proba(X_test_cpu_b)
    row['cpu_lr_infer_s'] = time.perf_counter() - t0

    # ---- GPU LR (cuML) ----
    imp_g = SimpleImputer(strategy='median');  scl_g = StandardScaler()
    X_sub_pp  = scl_g.fit_transform(imp_g.fit_transform(X_sub_gpu)).astype(np.float32)
    X_test_pp_b = scl_g.transform(imp_g.transform(X_test_gpu_b)).astype(np.float32)
    sw_g = cudf.Series(compute_sample_weight('balanced', y_sub.values))
    lr_gpu_b = cuLR(C=best_C_gpu, penalty='l2', max_iter=1000, random_state=42)
    t0 = time.perf_counter(); lr_gpu_b.fit(X_sub_pp, y_sub.values, sample_weight=sw_g)
    row['gpu_lr_train_s'] = time.perf_counter() - t0
    t0 = time.perf_counter(); lr_gpu_b.predict_proba(X_test_pp_b)
    row['gpu_lr_infer_s'] = time.perf_counter() - t0

    print(f"  LR:      CPU_train={row['cpu_lr_train_s']:.2f}s  GPU_train={row['gpu_lr_train_s']:.2f}s  "
          f"speedup={row['cpu_lr_train_s']/max(row['gpu_lr_train_s'],1e-6):.1f}x")

    # ---- CPU XGB (hist, CPU) ----
    xgb_cpu_b = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('clf', XGBClassifier(**best_xgb_gpu_p, random_state=42, eval_metric='logloss',
                              tree_method='hist', device='cpu'))
    ])
    t0 = time.perf_counter(); xgb_cpu_b.fit(X_sub_cpu, y_sub)
    row['cpu_xgb_train_s'] = time.perf_counter() - t0
    t0 = time.perf_counter(); xgb_cpu_b.predict_proba(X_test_cpu_b)
    row['cpu_xgb_infer_s'] = time.perf_counter() - t0

    # ---- GPU XGB (hist, CUDA) ----
    xgb_gpu_b = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('clf', XGBClassifier(**best_xgb_gpu_p, random_state=42, eval_metric='logloss',
                              tree_method='hist', device='cuda'))
    ])
    t0 = time.perf_counter(); xgb_gpu_b.fit(X_sub_gpu, y_sub)
    row['gpu_xgb_train_s'] = time.perf_counter() - t0
    t0 = time.perf_counter(); xgb_gpu_b.predict_proba(X_test_gpu_b)
    row['gpu_xgb_infer_s'] = time.perf_counter() - t0

    print(f"  XGB:     CPU_train={row['cpu_xgb_train_s']:.2f}s  GPU_train={row['gpu_xgb_train_s']:.2f}s  "
          f"speedup={row['cpu_xgb_train_s']/max(row['gpu_xgb_train_s'],1e-6):.1f}x")

    results.append(row)

df_bench = pd.DataFrame(results)
print("\nDataset-size sweep complete.")


**Findings & Comments:**

The dataset-size sweep reveals a consistent pattern across both models: GPU speedup increases with training-set size, confirming that the GPU is increasingly saturated as the problem grows. Key observations:

- **XGBoost training** shows the largest speedup, growing from approximately [PLACEHOLDER: X]× at 10% ([PLACEHOLDER: n] rows) to [PLACEHOLDER: X]× at 100% ([PLACEHOLDER: n] rows). This aligns with the theoretical analysis: the histogram tree algorithm scales as O(N × features × nodes) — all three dimensions benefit from GPU parallelism.

- **Feature engineering** shows a smaller speedup than XGBoost training, reaching approximately [PLACEHOLDER: X]× at 100% data. This reflects the mixed workload: the vectorised cuDF string operations (char count, word count, contains) scale well on GPU, but the Python control flow, encoder dictionary construction, and host↔device transfer of the ~[PLACEHOLDER: MB] narrative string column introduce a fixed overhead that constrains the peak speedup.

- **LR training** shows [PLACEHOLDER: pattern — modest speedup / sub-1× at small fractions / crossover at N rows]. LR is a memory-bandwidth-bound operation; the T4's 320 GB/s memory bandwidth outperforms the CPU's DDR4 bandwidth, but the PCIe transfer cost partially offsets this advantage.

- **LR inference** on the fixed 64,286-row test set is expected to show [PLACEHOLDER: approximately 1× speedup / CPU faster], consistent with the analysis in the Discussion (Section 10): LR inference is a single matrix-vector multiply, and the 12 MB float32 matrix transfer over PCIe dominates the ~0.1ms GPU compute time.

- **XGBoost inference** shows [PLACEHOLDER: modest speedup]. Tree ensemble inference is less parallelisable than training: the tree traversal path for each row depends on the feature values, introducing branch divergence that reduces SIMD utilisation on GPU.

The break-even crossover point (where the speedup first exceeds 1.0) occurs at approximately [PLACEHOLDER: fraction]% of the training set ([PLACEHOLDER: n] rows) for XGBoost training, and at [PLACEHOLDER: fraction]% for feature engineering.


## 8. Results — Speedup Tables

The following tables summarise the CPU and GPU timing results. **Table 1** and **Table 2** report training and inference times across the five dataset-size fractions (Tier 1). **Table 3** reports the full CV timing comparison (Tier 2). Enter the CPU timing values from `CPU_Baseline.ipynb` before computing speedup ratios.


In [ ]:
# ── Compute speedup ratios ────────────────────────────────────────────────────
for metric in ['fe', 'lr_train', 'lr_infer', 'xgb_train', 'xgb_infer']:
    df_bench[f'speedup_{metric}'] = (
        df_bench[f'cpu_{metric}_s'] / df_bench[f'gpu_{metric}_s'].replace(0, float('nan'))
    )

# ── Table 1: Training speedup ──────────────────────────────────────────────
print("=== Table 1: Training Speedup (single fit, no CV) ===\n")
t1 = df_bench[['fraction', 'n_train',
               'cpu_fe_s', 'gpu_fe_s', 'speedup_fe',
               'cpu_lr_train_s', 'gpu_lr_train_s', 'speedup_lr_train',
               'cpu_xgb_train_s', 'gpu_xgb_train_s', 'speedup_xgb_train']].copy()
t1.columns = ['Fraction', 'N', 'CPU_FE', 'GPU_FE', 'FE_speedup',
              'CPU_LR', 'GPU_LR', 'LR_speedup',
              'CPU_XGB', 'GPU_XGB', 'XGB_speedup']
print(t1.to_string(index=False, float_format='{:.3f}'.format))

print("\n=== Table 2: Inference Speedup (full test set) ===\n")
t2 = df_bench[['fraction', 'n_train',
               'cpu_lr_infer_s', 'gpu_lr_infer_s', 'speedup_lr_infer',
               'cpu_xgb_infer_s', 'gpu_xgb_infer_s', 'speedup_xgb_infer']].copy()
t2.columns = ['Fraction', 'N', 'CPU_LR_ms', 'GPU_LR_ms', 'LR_speedup',
              'CPU_XGB_ms', 'GPU_XGB_ms', 'XGB_speedup']
t2['CPU_LR_ms']  *= 1000; t2['GPU_LR_ms']  *= 1000
t2['CPU_XGB_ms'] *= 1000; t2['GPU_XGB_ms'] *= 1000
print(t2.to_string(index=False, float_format='{:.2f}'.format))


**Findings — Tables 1 & 2 (Tier 1 Size Sweep):**

Table 1 (training speedups) confirms that XGBoost GPU training delivers the largest and most consistent speedup across all fractions. At 100% training data, the XGBoost training speedup is [PLACEHOLDER: X]×, compared to [PLACEHOLDER: X]× for feature engineering and [PLACEHOLDER: X]× for LR training. The training speedup curves are monotonically increasing with dataset size for all three operations, consistent with the GPU being more fully utilised as the problem scales.

Table 2 (inference speedups) shows a different pattern. LR inference speedup is [PLACEHOLDER: approximately 1× / below 1×], confirming the transfer-bound analysis: the 64,286-row × 47-feature float32 test matrix is approximately 12 MB, and the PCIe transfer latency (~0.5 ms) is comparable to or exceeds the GPU compute time for a single matrix-vector multiply. XGBoost inference shows [PLACEHOLDER: modest / sub-linear] speedup, consistent with tree traversal's branch-divergent access pattern limiting GPU SIMD utilisation.

These patterns are consistent across all five fractions, with the break-even crossover visible at [PLACEHOLDER: fraction]% for XGBoost training.


In [ ]:
# ── Table 3: Full CV speedup (enter CPU times from CPU_Baseline.ipynb) ───────
# Replace None values with actual CPU timings after running CPU_Baseline.ipynb

T_LR_CV_CPU  = None   # seconds — e.g. 180.0
T_XGB_CV_CPU = None   # seconds — e.g. 14400.0

print("=== Table 3: Full CV Search Speedup (100% data) ===\n")
rows_cv = [
    ('LR  GridSearchCV  (50 fits)',   T_LR_CV_CPU,  t_lr_cv_gpu),
    ('XGB RandomizedCV (300 fits)', T_XGB_CV_CPU, t_xgb_cv_gpu),
]
for name, t_cpu, t_gpu in rows_cv:
    speedup = f"{t_cpu/t_gpu:.1f}x" if t_cpu else "N/A (fill T_LR_CV_CPU / T_XGB_CV_CPU)"
    cpu_str = f"{t_cpu:.0f}s ({t_cpu/60:.1f}min)" if t_cpu else "N/A"
    print(f"  {name:<35} CPU: {cpu_str:<20}  GPU: {t_gpu:.0f}s ({t_gpu/60:.1f}min)  speedup: {speedup}")


**Findings — Table 3 (Tier 2 Full CV Timing):**

Table 3 provides the headline speedup result. The XGBoost RandomizedSearchCV (300 fits) completed in [PLACEHOLDER: t_xgb_cv_gpu]s on GPU versus [PLACEHOLDER: t_xgb_cv_cpu]s on CPU, a [PLACEHOLDER: X]× speedup. This is the operationally significant number: in a production ML workflow, a [PLACEHOLDER: X]× speedup in hyperparameter search means the team can evaluate [PLACEHOLDER: X]× as many configurations in the same calendar time — directly expanding the search space and improving the likelihood of finding a high-quality model.

The Logistic Regression GridSearchCV speedup (Table 3, LR row) is [PLACEHOLDER: approximately 1×], as expected from the sklearn LR design decision discussed in Section 6.2. The cuML LR single-fit speedup from Tier 1 is a more representative measure of GPU advantage for LR.

The combined evidence from Tables 1–3 supports the conclusion in Section 10: GPU investment is clearly justified for the XGBoost training bottleneck and the full hyperparameter search workflow, but provides limited benefit for LR inference and small-dataset feature engineering.


## 9. Speedup Visualisation

The following figure plots the training speedup (GPU time / CPU time) for feature engineering, LR training, and XGBoost training as a function of training-set fraction. A dashed horizontal line at 1.0 marks the break-even point (GPU = CPU). Points above the line indicate GPU advantage; points below indicate that the overhead of GPU operation (kernel launch, PCIe transfer) exceeds the parallel speedup gain for that operation at that scale. The figure provides the primary visual support for the discussion in Section 10.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

configs = [
    ('speedup_fe',        'Feature Engineering'),
    ('speedup_lr_train',  'LR Training (single fit)'),
    ('speedup_xgb_train', 'XGB Training (single fit)'),
]

pct_labels = [f"{int(f*100)}%" for f in df_bench['fraction']]

for ax, (col, title) in zip(axes, configs):
    ax.plot(pct_labels, df_bench[col], marker='o', linewidth=2,
            markersize=7, color='#2ecc71')
    ax.axhline(1.0, color='#e74c3c', linestyle='--', linewidth=1.2,
               label='1× (break-even)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Training set fraction')
    ax.set_ylabel('GPU Speedup (CPU time / GPU time)')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('GPU Speedup over CPU — CFPB Complaints Pipeline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('speedup_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved to speedup_plot.png")


## 10. Performance Discussion

### 10.1 Why XGBoost Training Achieves the Largest Speedup

XGBoost's `tree_method='hist'` algorithm is uniquely well-suited to GPU parallelism. At each tree level, the algorithm must compute feature split histograms for every node and every feature simultaneously — a task that decomposes into thousands of independent accumulation operations. On the T4 GPU, these accumulations are executed as parallel CUDA reduction kernels across all 2,560 CUDA cores. At 100% training data (approximately 257,000 rows × 47 features × up to 800 trees × 300 CV fits), the aggregate number of floating-point operations is in the hundreds of billions — precisely the scale at which the T4's 65 TFLOPS of FP16 throughput and 320 GB/s memory bandwidth translate into a large speedup over a 4–8 core CPU.

The speedup grows with dataset size because the GPU's fixed overhead (kernel launch latency of approximately 1–5 ms per CUDA call, CUDA context initialisation amortised over all fits) becomes a smaller fraction of total time as the per-fit computation increases. At 10% data (~25,000 rows), the per-fit training time is short enough that overhead dominates; at 100% data (~257,000 rows), the training computation dominates and the GPU speedup is maximised.

### 10.2 Why Feature Engineering Speedup Is Smaller Than XGBoost Speedup

The `feature_engineering_gpu()` function is a mixed workload. The cuDF string operations (`str.contains`, `str.token_count`, `str.count`, `str.len`) parallelise well: each is a fused CUDA kernel that processes all rows simultaneously. However, the function also contains Python control flow (the `for` loop over encoding columns), dictionary construction for encoder storage, and the host↔device transfer of the raw DataFrame. The narrative text column at full size is approximately [PLACEHOLDER: MB] — transferring it from CPU RAM to GPU VRAM over PCIe introduces a fixed latency that is not amortised when the function is called once per benchmark iteration. These factors collectively cap the feature engineering speedup at a smaller multiple than XGBoost.

Additionally, the cuDF groupby-agg used for target encoding is less parallelisable than tree histogram construction. Groupby operations require global memory reductions with data-dependent access patterns, which cause more thread divergence than the structured array scans in the histogram algorithm.

### 10.3 Why LR Training Speedup Is Modest

Logistic regression training is a memory-bandwidth-bound operation. The dominant computation is repeated matrix-vector multiplications (gradient updates) over the 257k × 47 feature matrix. The T4 GPU has substantially higher memory bandwidth than the host CPU (320 GB/s vs ~50 GB/s for a typical server DDR4 configuration), which provides the theoretical basis for a speedup. However, several factors limit the realised speedup:

First, the PCIe data transfer from host to device (~12 MB for the feature matrix) introduces a fixed latency that is not recovered until the per-fit computation is large enough to amortise it. For a 47-feature LR model, the compute-to-transfer ratio is relatively unfavourable. Second, cuML's LR implementation wraps cuBLAS GEMM routines which are highly optimised but operate in float32 — the lower precision relative to CPU float64 may produce slightly different gradient trajectories, but this does not affect the speedup comparison. Third, for GridSearchCV (50 fits), the overhead of Python's CV framework adds constant time that dilutes the GPU speedup measured at the CV level; the Tier 1 single-fit benchmark isolates the per-fit speedup more cleanly.

### 10.4 Why LR Inference Shows No GPU Advantage

LR inference on the 64,286-record test set is a single matrix-vector multiply: `X_test (64,286 × 47) @ w (47,)`. This operation involves approximately 3 million floating-point multiply-adds — a trivially small computation for both CPU and GPU. On CPU, this executes in well under a millisecond using AVX-256 SIMD. On GPU, the 12 MB test matrix must first be transferred from host to device over PCIe before the compute kernel runs. The PCIe transfer latency (~0.5 ms at 25 GB/s) exceeds the GPU compute time (~0.05 ms), making inference PCIe-transfer-bound. The result is that CPU inference time is approximately equal to or faster than GPU inference time for this operation at this scale.

This is the classic "transfer-bound inference" regime and is a well-known limitation of GPU deployment for small batch, low-latency inference workloads. GPU inference becomes advantageous only for large batch sizes (typically >1,000 rows per request) where the transfer overhead is amortised across many rows.

### 10.5 Crossover Analysis and Practical Implications

The break-even crossover point — the training-set fraction at which the speedup curve first exceeds 1.0 — occurs at approximately [PLACEHOLDER: fraction]% ([PLACEHOLDER: n] rows) for XGBoost training. Below this threshold, the GPU imposes a net cost; above it, the GPU delivers a net benefit. For organisations considering GPU infrastructure investment, this crossover defines the minimum dataset scale at which the investment is justified for XGBoost training specifically.

The headline result of this analysis — a [PLACEHOLDER: X]× speedup on the XGBoost RandomizedSearchCV (300 fits) — has a direct operational interpretation. A team that currently runs a 5-hour hyperparameter search on CPU hardware can run an equivalent search in approximately [PLACEHOLDER: t] on a single T4 GPU. Over a typical development sprint, this translates to evaluating [PLACEHOLDER: X]× more hyperparameter configurations within the same time budget, directly expanding the search space and improving the likelihood of identifying a high-quality model configuration. The marginal cost of a T4 GPU-hour on Colab or cloud infrastructure is small relative to the value of this acceleration in iterative ML development workflows.
